In [ ]:
# tudo precisa estar na mesma ordem
# dado que o que foi subido para o faiss está ordenado e está relacionado a outras estruturas de dados ordenadas
# isso tudo segundo um mesmo pareamento

# faiss retorna o índice global(índice que ele recebeu na carga das hashes)
# precisamos mapear esse índice global para o índice local das nossas estruturas de dados

# precisamos manter uma lista com as bordas inferiores referentes aos hashes de cada fingerprint armazenada no faiss
# toda vez que novas hashes são adicionadas ao faiss, pegamos o total de hashes do índice do faiss e definimos como uma nova borda
# essa lista de bordas estará pareada com uma lista de recordids

# assim, dado um índice global retornado pelo faiss, podemos recupera o recordid correspondente
# e então recuperar a fingerprint completa

# a partir disso podemos realizar a funções de filtragem, validação...

## Store

In [ ]:
from qfp.fingerprint import ReferenceFingerprint

In [ ]:
fp = ReferenceFingerprint.load_from_pickle1("/mnt/disk1/BAF/qfp_features/references/ref_0003_fingerprint.pkl")

In [ ]:
# fp = {
#     "peaks": array([[   59,   465],
#        [   61,   410],
#        [   65,   305],
#        ...,
#        [33431,   212],
#        [33461,   451],
#        [33479,   308]])
#     "strongest": array([[   72,    50,   390, ...,    67,   391,   407],
#        [   72,    50,   390, ...,    67,   491,   464],
#        [   72,    50,   390, ...,    67,   494,   362],
#        ...,
#        [33156,   134, 33385, ...,   308, 33479,   308],
#        [33156,   134, 33385, ...,   451, 33461,   451],
#        [33156,   134, 33407, ...,   212, 33479,   308]]),
#     "hashes": array([[0.9968652 , 0.04761905, 0.9968652 , 0.04761905],
#        [0.75894988, 0.0410628 , 0.75894988, 0.0410628 ],
# }

In [ ]:
def store(self, fp, cod_fonogram):
    recordid = cod_fonogram
    self.fingerprints = {}

    self.fingerprints[cod_fonogram] = {
        'recordid': recordid,
        'peaks': fp.peaks,
        'quads': fp.strongest,
        'hashes': fp.hashes
    }

    n_vectors = self.faiss_index.ntotal
    start = n_vectors

    self.faiss_index.add(fp.hashes)

    self.border_list.append(start)
    self.recordid_list.append(recordid)

## Query

In [ ]:
@njit(cache=True, debug=True)
def _filter_candidates_core(quads_query, quads_candidates, e_tolerance=0.2):

    # qQuads_arr assumed float64: Ax,Ay,Bx,By,Cx,Cy,Dx,Dy

    qAx, qAy, qBx, qBy = quads_query

    # iterate indices in I[start:end]
    for quad in quads_candidates:
        # recupera cQuad dos arrays; todos inteiros 
        cAx, cAy, cBx, cBy = quad

        # Rough pitch coherence:
        #   1/(1+e) <= queAy/canAy <= 1/(1-e)
        if cAy == 0:
            continue
        ratio = qAy / cAy
        if not (1.0 / (1.0 + e_tolerance) <= ratio <= 1.0 / (1.0 - e_tolerance)):
            continue

        # X transformation tolerance check:
        #   sTime = (queBx-queAx)/(canBx-canAx)
        denom = (cBx - cAx)
        if denom == 0:
            continue
        sTime = (qBx - qAx) / denom
        if not (1.0 / (1.0 + e_tolerance) <= sTime <= 1.0 / (1.0 - e_tolerance)):
            continue

        # Y transformation tolerance check:
        #   sFreq = (queBy-queAy)/(canBy-canAy)
        denom2 = (cBy - cAy)
        if denom2 == 0:
            continue
        sFreq = (qBy - qAy) / denom2
        if not (1.0 / (1.0 + e_tolerance) <= sFreq <= 1.0 / (1.0 - e_tolerance)):
            continue

        # Fine pitch coherence:
        #   |queAy-canAy*sFreq| <= eFine
        # Obs: qAy e cAy são floats/integer; operação segura
        if abs(qAy - (cAy * sFreq)) > 1.8:
            continue

        # offset
        offset = cAx - (qAx / sTime)

        # append em typed lists
        recordids.append(recordid)
        offsets.append(offset)
        sTimes.append(sTime)
        sFreqs.append(sFreq)
return recordids, offsets, sTimes, sFreqs

In [ ]:
# retorna limites(lims) para cada query(hash)
# os limites são usados para identificar quais indices(I) pertencem a cada query

def query(self, fp_query, radius=0.2):

    lims, D, I = self._faiss_batch_search(fp_query.hashes, radius)
    for start, end in zip(lims[:-1], lims[1:]):
        faiss_index = I[start:end] 

        for idx_faiss, quads_query in zip(faiss_index, fp_query.strongest):

            # 1) Encontrar os intervalos de cada índice
            interval_idx = np.searchsorted(self.border_list, idx_faiss, side='right') - 1

            # 2) Pegar o musicid correspondente a cada índice
            list_recordid_ref = self.recordid_list[interval_idx]

            # 3) Pegar a posição relativa dentro do intervalo
            list_musicid_index_ref = idx_faiss - self.border_list[interval_idx]

            list_quads_ref = self.fingerprints[list_recordid_ref]['strongest'][list_musicid_index_ref]

            filtered = self._filter_candidates(quads_query, list_quads_ref, list_recordid_ref e_tolerance=0.2)

In [ ]:
        for start, end in zip(lims[:-1], lims[1:]):
            faiss_index_list = I[start:end] 

            for idx_faiss, quads_query in zip(faiss_index_list, fp_query.strongest):

                # 1) Encontrar os intervalos de cada índice
                interval_idx = np.searchsorted(self.border_list, idx_faiss, side='right') - 1
                print("interval_idx: ", interval_idx)

                # 2) Pegar o musicid correspondente a cada índice
                list_recordid_ref = self.phonogram_code_list[interval_idx]
                print("list_recordid_ref: ", list_recordid_ref)

                # 3) Pegar a posição relativa dentro do intervalo
                list_musicid_index_ref = idx_faiss - self.border_list[interval_idx]
                print("list_musicid_index_ref: ", list_musicid_index_ref)

                list_quads_ref = self.fingerprints[list_recordid_ref]['strongest'][list_musicid_index_ref]
                print("list_quads_ref: ", list_quads_ref)
                print("list_quads_ref len: ", len(list_quads_ref))

In [ ]:
        for idx_faiss in faiss_index:
            # 1) Encontrar os intervalos de cada índice
            interval_idx = np.searchsorted(self.border_list, idx_faiss, side='right') - 1

            # 2) Pegar o musicid correspondente a cada índice
            # list_phonogram_code_ref = self.phonogram_code_list[interval_idx]
            list_phonogram_code_ref = [self.phonogram_code_list[i] for i in interval_idx]

            # 3) Pegar a posição relativa dentro do intervalo
            # list_musicid_index_ref = idx_faiss - self.border_list[interval_idx]
            list_phonogram_code_index_ref = [i - self.border_list[j] for i, j in zip(idx_faiss, interval_idx)]

            # list_quads_ref = self.fingerprints[list_phonogram_code_ref]['strongest'][list_phonogram_code_index_ref]
            list_quads_ref = [
                self.fingerprints[pc]['strongest'][idx]
                for pc, idx in zip(list_phonogram_code_ref, list_phonogram_code_index_ref)
            ]
            candidates_quads_all.append(list_quads_ref)
            candidates_phonogram_code_all.append(list_phonogram_code_ref)

        with open("candidates.txt", "w") as f:
            f.write("candidates_quads_all: \n")
            for item in candidates_quads_all:
                f.write(f"{item}\n")
            f.write("\ncandidates_phonogram_code_all: \n")
            for item in candidates_phonogram_code_all:
                f.write(f"{item}\n")
        # 2. Aplicar filtros nos resultados
        filter_start = time.time()
        filtered = self._filter_candidates(fp_query.strongest, lims, I, e_tolerance=0.2)
        filter_end = time.time()

In [ ]:
# filtered = self._filter_candidates(quads_query, list_quads_ref, list_recordid_ref e_tolerance=0.2)